In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [2]:
loader = PyPDFLoader(
    file_path='../data/securely.pdf',
    )

In [3]:
doc = loader.load()

In [4]:
len(doc)

59

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
splitted_data = splitter.split_documents(doc)

In [13]:
len(splitted_data)

66

In [14]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

In [15]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
)

In [21]:
query = "User manual content"
data = vector_store.similarity_search(query=query)
len(data)

4

In [ ]:
context = ''

for doc in data:
    context += doc.page_content + '\n'

In [20]:
llm = ChatOpenAI(model='gpt-5-nano-2025-08-07')

In [22]:
res = llm.invoke(f'''Can you provide me answer based on 
                 provided context for my questions, context: {context} and question: {query}''')

In [23]:
print(res.content)

Here is the User Manual content based on the provided context.

CHAPTER 5: USER MANUAL

1. Integration Setup
- Add the package to your Flutter project's pubspec.yaml dependencies:
  dependencies:
    securely: ^1.1.0
- Ensure platform configurations are set up:
  - iOS/macOS: Verify sandbox keychain sharing permissions are configured inside Xcode.
  - Android: Ensure minimum SDK version is set to 21 inside android/app/build.gradle.

2. Code Examples & Customization
- Scenario A: Telemetry Checks & RASP
- Scenario B: SecureTextField & SecureKeyboard
- Scenario C: StoreSecurely

3. App Screenshots

If you’d like, I can format this into a clean, print-ready document or fill in details for Scenarios B and C if you provide their content.


### Chain -> Context generate | prompt | llm | strparser

In [25]:
def get_context(query: str):
    data = vector_store.similarity_search(query=query)

    context = ''
    for doc in data:
        context += doc.page_content + '\n'

    return {
        'context': context,
        'question': query
    }

In [ ]:
prompt = PromptTemplate.from_template(
    '''
    You are a helpful assistant and provide answer based on the context for user question, and
    if you don't know the answer, then you can say that 'I don't know.'
    Context: {context}
    Question: {question}
'''
)

In [27]:
rag_chain = get_context | prompt | llm

In [28]:
res = rag_chain.invoke('Tell me about securely')

In [29]:
print(res.content)

Securely is a cross-platform Flutter package that bundles multiple security features into one unified solution for mobile, desktop, and web apps. It’s designed to address the fragmented Flutter security ecosystem by providing a single API for core protection layers.

Key components and capabilities:

- Runtime Application Self-Protection (RASP)
  - Real-time detection of debuggers, root/jailbreak status, emulators/simulators
  - Detection of Frida instrumentation, VPN usage, developer mode, USB debugging, and screen recording/casting
  - Telemetry checks that use low-level OS commands (e.g., sysctl on iOS, root checks on Android) to identify environmental threats

- Hardware-backed secure storage
  - StoreSecurely for cryptographic storage
  - Supports configurable AES-GCM and AES-CBC encryption
  - Uses native platform keystores like Android Keystore and iOS Keychain
  - Provides cryptographic isolation at the hardware level to protect data from cold-boot RAM scans and unauthorized ac

In [35]:
res = rag_chain.invoke('how many functional requirements')
print(res.content)

There are 9 functional requirements: FR-04, FR-05, FR01, FR02, FR-03, FR-18, FR-19, FR-20, FR-21.
